In [0]:
import time
import numpy as np
import pandas as pd
import mlflow
from datetime import datetime, timezone

t0 = time.time()
run_stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

def robust_z(change, window, min_obs):
    prior = change.shift(1)
    med = prior.rolling(window, min_periods=min_obs).median()
    mad = prior.rolling(window, min_periods=min_obs).apply(lambda a: np.median(np.abs(a - np.median(a))), raw=True)
    return (change - med) / (1.4826 * mad.replace(0, np.nan))

series = []   # (name, group, frequency, dates, value, change, threshold)

# Daily prices (log returns) - keep both for the cross-check
p = spark.sql("""SELECT price_date, series_id, price FROM oil_shock.gold.fct_price_daily
                 WHERE series_id IN ('RBRTE', 'RWTC') AND price_date >= '2015-01-01'""").toPandas()
p["price_date"] = pd.to_datetime(p["price_date"])
rets = {}
for sid, label in [("RBRTE", "Brent physical"), ("RWTC", "WTI spot")]:
    s = p[p["series_id"] == sid].sort_values("price_date")
    r = np.log(s["price"].where(s["price"] > 0)).diff()
    rets[label] = pd.Series(r.values, index=s["price_date"].values)
    series.append((label, "Prices", "daily", s["price_date"].values, s["price"].values, r.values, 4.0))

sp = spark.table("oil_shock.gold.fct_spread_daily").toPandas()
sp["price_date"] = pd.to_datetime(sp["price_date"])
sp = sp[sp["price_date"] >= "2015-01-01"].sort_values("price_date")
for col, label in [("brent_wti_spread", "Brent-WTI spread"), ("diesel_crack", "Diesel crack")]:
    s = sp.dropna(subset=[col])
    series.append((label, "Spreads", "daily", s["price_date"].values, s[col].values, s[col].diff().values, 4.0))

ck = spark.sql("""SELECT transit_date, chokepoint_name, ships_tanker_pct_of_2025 AS pct
                  FROM oil_shock.gold.fct_chokepoint_daily
                  WHERE lower(chokepoint_name) LIKE '%hormuz%' OR lower(chokepoint_name) LIKE '%mandeb%'
                     OR lower(chokepoint_name) LIKE '%suez%'""").toPandas()
ck["transit_date"] = pd.to_datetime(ck["transit_date"])
for key, label in [("hormuz", "Hormuz tanker transits"), ("mandeb", "Bab el-Mandeb tanker transits"),
                   ("suez", "Suez tanker transits")]:
    s = ck[ck["chokepoint_name"].str.lower().str.contains(key)].dropna(subset=["pct"]).sort_values("transit_date")
    series.append((label, "Flows", "daily", s["transit_date"].values, s["pct"].values, s["pct"].diff().values, 4.0))

wk = spark.table("oil_shock.gold.fct_us_weekly_balance").select("week_ending", "crude_stocks_change_kbbl").toPandas()
wk["week_ending"] = pd.to_datetime(wk["week_ending"])
wk = wk.dropna().sort_values("week_ending")
series.append(("US crude stock change", "Inventories", "weekly", wk["week_ending"].values,
               wk["crude_stocks_change_kbbl"].values, wk["crude_stocks_change_kbbl"].values, 3.0))

pos = spark.sql("""SELECT report_date, contract_key, report_type, managed_money_net_change
                   FROM oil_shock.gold.fct_positioning_weekly
                   WHERE contract_key IN ('brent_ice', 'wti_nymex')""").toPandas()
pos["report_date"] = pd.to_datetime(pos["report_date"])
pos["rank"] = (pos["report_type"] != "futures_only").astype(int)
pos = pos.sort_values(["contract_key", "report_date", "rank"]).drop_duplicates(["contract_key", "report_date"])
for key, label in [("brent_ice", "Brent fund positioning change"), ("wti_nymex", "WTI fund positioning change")]:
    s = pos[pos["contract_key"] == key].dropna(subset=["managed_money_net_change"]).sort_values("report_date")
    series.append((label, "Positioning", "weekly", s["report_date"].values, s["managed_money_net_change"].values,
                   s["managed_money_net_change"].values, 3.0))

# ---------- Score ----------
frames = []
for name, group, freq, dates, value, change, thr in series:
    d = pd.DataFrame({"date": pd.to_datetime(dates), "value": value, "change": change})
    window, min_obs = (60, 30) if freq == "daily" else (52, 26)
    d["robust_z"] = robust_z(d["change"], window, min_obs)
    d["next_change"] = d["change"].shift(-1)
    d["series"], d["group"], d["frequency"], d["threshold"] = name, group, freq, thr
    frames.append(d)

rk = spark.sql("""SELECT news_date AS date, oil_risk_zscore AS value FROM oil_shock.gold.fct_risk_index_daily
                  WHERE event_region = 'Gulf' AND NOT is_data_gap AND oil_risk_zscore IS NOT NULL""").toPandas()
rk["date"] = pd.to_datetime(rk["date"])
rk["change"], rk["robust_z"], rk["next_change"] = np.nan, rk["value"], np.nan
rk["series"], rk["group"], rk["frequency"], rk["threshold"] = "Gulf risk surprise", "Geopolitics", "daily", 3.0
frames.append(rk)

allz = pd.concat(frames, ignore_index=True)
flag = allz[allz["robust_z"].abs() >= allz["threshold"]].copy()

# "Check data" rule (prices only): spike reverses >= 70% next day AND the other benchmark moved < half as much
other = {"Brent physical": "WTI spot", "WTI spot": "Brent physical"}
def other_ret(row):
    if row["series"] not in other:
        return np.nan
    return rets[other[row["series"]]].get(row["date"], np.nan)
flag["other_benchmark_change"] = flag.apply(other_ret, axis=1)
is_price = flag["group"] == "Prices"
reversal = (-flag["next_change"] / flag["change"]) >= 0.7
idiosyncratic = flag["other_benchmark_change"].abs() < 0.5 * flag["change"].abs()
flag["flag_type"] = np.where(is_price & reversal & idiosyncratic, "check data (reversed next day, one series only)",
                             "unusual move")
flag["severity"] = np.where(flag["robust_z"].abs() >= 6, "extreme", "notable")
flag["period"] = np.where(flag["date"] >= "2026-02-28", "war", "pre_war")
flag = flag[["date", "series", "group", "frequency", "value", "change", "robust_z", "other_benchmark_change",
             "flag_type", "severity", "period"]]
flag["robust_z"] = flag["robust_z"].round(2)

latest = (allz.dropna(subset=["robust_z"]).sort_values("date").groupby("series").tail(1)
          [["series", "group", "date", "value", "robust_z", "threshold"]].copy())
latest["is_flagged"] = latest["robust_z"].abs() >= latest["threshold"]
latest["robust_z"] = latest["robust_z"].round(2)

for nm, out in {"anomaly_flags": flag, "anomaly_latest": latest}.items():
    out = out.copy(); out["run_at_utc"] = run_stamp
    spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"oil_shock.ml.{nm}")

counts = flag.groupby(["series", "period"]).size().unstack(fill_value=0)
checks = flag[flag["flag_type"] != "unusual move"]
mlflow.set_experiment("/Users/zulhaq2@gmail.com/oil-shock-anomalies")
with mlflow.start_run(run_name=f"anomaly_flags v2 {run_stamp}"):
    mlflow.set_tags({"notebook": "notebooks/04_ml/45_anomaly_flags", "project": "oil-shock-2026", "author": "Zia Malik"})
    mlflow.log_params({"method": "robust z (median/MAD) of change vs prior window",
                       "daily": "window 60, min 30, threshold 4", "weekly": "window 52, min 26, threshold 3",
                       "check_data_rule": "price flag; next day reverses >= 70%; other benchmark moved < 50%"})
    mlflow.log_metrics({"flags_total": len(flag), "flags_war": int((flag["period"] == "war").sum()),
                        "check_data_flags": len(checks)})

# ---------- Compact summary to paste ----------
print(f"Done in {time.time() - t0:.0f}s | flags: {len(flag)} | check-data flags: {len(checks)}")
print(counts.to_string())
print("\nCheck-data flags (prices):")
print(checks[["date", "series", "value", "change", "other_benchmark_change", "robust_z"]].round(3).to_string(index=False))